## Load Data

For our exercises we will use two different datasets:
- The AG News subset that features english news articles of 4 different categories
    - https://www.kaggle.com/datasets/amananandrai/ag-news-classification-dataset
- The 10kGNAD dataset that features german news articles of 9 different categories
    - https://www.kaggle.com/datasets/mexwell/10kgnad

In [ ]:
import csv
import pandas as pd
from typing import List, Set, Tuple

# english data
classes_en = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
train_en = pd.read_csv("../data/AGNews/train.csv", 
                       names = ["Label", "Title", "Article"],
                       encoding = "utf-8")
test_en = pd.read_csv("../data/AGNews/test.csv", 
                      names = ["Label", "Title", "Article"],
                      encoding = "utf-8")

# german data
train_de = pd.read_csv("../data/10kGNAD/train.csv", 
                       sep = ";", names = ["Label", "Article"], 
                       quotechar = "\'", quoting = csv.QUOTE_MINIMAL, encoding = "utf-8")
test_de = pd.read_csv("../data/10kGNAD/test.csv", 
                       sep = ";", names = ["Label", "Article"], 
                       quotechar = "\'", quoting = csv.QUOTE_MINIMAL, encoding = "utf-8")

We can iterate of the dataframe cols to construct a custom list of documents to work on

In [6]:
labels_en = [classes_en[int(row["Label"])] for i, row in train_en.iterrows()]
articles_en = [row["Article"] for i, row in train_en.iterrows()]
labels_de = [row["Label"] for i, row in train_de.iterrows()]
articles_de = [row["Article"] for i, row in train_de.iterrows()]

In [7]:
articles_en[:5]


["Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'Reuters - Private investment firm Carlyle Group,\\which has a reputation for making well-timed and occasionally\\controversial plays in the defense industry, has quietly placed\\its bets on another part of the market.',
 'Reuters - Soaring crude prices plus worries\\about the economy and the outlook for earnings are expected to\\hang over the stock market next week during the depth of the\\summer doldrums.',
 'Reuters - Authorities have halted oil export\\flows from the main pipeline in southern Iraq after\\intelligence showed a rebel militia could strike\\infrastructure, an oil official said on Saturday.',
 'AFP - Tearaway world oil prices, toppling records and straining wallets, present a new economic menace barely three months before the US presidential elections.']

## NLTK Overview

https://www.nltk.org/ is a leading Python platform for working with human language data. It provides:

- Interfaces to 50+ corpora and lexical resources (e.g., WordNet)
- Text processing libraries for:
  - Tokenization
  - Stemming
  - Tagging
  - Parsing
  - Semantic reasoning
- Wrappers for industrial-strength NLP libraries

NLTK is widely used for research, education, and prototyping in natural language processing.

---

### What We Will Do

We will preprocess English and German articles using NLTK by applying:

1. Tokenization – Splitting text into words or sentences.
2. Stemming – Reducing words to their root form.
3. Stopword Removal – Removing common words that carry little meaning.

After each step, we’ll inspect how the text changes, so you can see the transformation clearly.

This mirrors the steps we previously saw in SpaCy, where much of this was automated, but here we’ll perform them manually to understand each process in detail and see how NLTK can be used.

In [8]:
# import required packages
import nltk 
from nltk.corpus import stopwords as nltkStopwords
from nltk.stem.snowball import SnowballStemmer

### NLTK tokenizes documents which are any string variables

In [9]:
# down load nltk resources 
nltk.download("punkt") #  tokenizer
nltk.download("punkt_tab") 
nltk.download("stopwords") # stopword list 


# tokenize the document
# contrary to spacey, nltk requires to tokenize each document separately
articles_en_tokenized = [nltk.word_tokenize(doc) for doc in articles_en]
articles_de_tokenized = [nltk.word_tokenize(doc) for doc in articles_de]

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\p42011\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\p42011\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\p42011\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [10]:
articles_en_tokenized[0]

['Reuters',
 '-',
 'Short-sellers',
 ',',
 'Wall',
 'Street',
 "'s",
 'dwindling\\band',
 'of',
 'ultra-cynics',
 ',',
 'are',
 'seeing',
 'green',
 'again',
 '.']

### Stemming can be done with NLTK's Snowball Stemming

[https://www.nltk.org/api/nltk.stem.snowball.html](https://www.nltk.org/api/nltk.stem.snowball.html)

We will write a function for this to loop over each article of our list

In [11]:
# we need to define a function to apply the snowballstemmer to our document
def stem(tokenized_document: str, language: str | None = None) -> List[str]:
    # Initialize the Snowball stemmer for the specified language
    # `ignore_stopwords=False` ensures stopwords are not excluded from stemming
    stemmer = SnowballStemmer(language, ignore_stopwords=False)
    
    # Apply stemming to each word in the tokenized document
    # Assumes `tokenized_document` is a list of words (not a raw string)
    return [stemmer.stem(word) for word in tokenized_document]

# again, we need to loop over each document in the list
articles_en_stemmed = [stem(doc, "english") for doc in articles_en_tokenized]
articles_de_stemmed = [stem(doc, "german") for doc in articles_de_tokenized]

# spacey equivalent:
# doc = nlp(articles_en[0])
# tokens = [token.lemma_ for token in doc]


NLTK also offers built-in stopword sets for different languages

In [12]:
articles_en_stemmed[0]


['reuter',
 '-',
 'short-sel',
 ',',
 'wall',
 'street',
 "'s",
 'dwindling\\band',
 'of',
 'ultra-cyn',
 ',',
 'are',
 'see',
 'green',
 'again',
 '.']

The english stopwords are: 

In [13]:
stopwords_en = set(nltkStopwords.words("english"))
stopwords_de = set(nltkStopwords.words("german"))

In [14]:
# join the stopwords with a ", " and print them
", ".join(stopwords_en)

"own, i, it'll, having, ma, where, we'd, against, ourselves, doesn't, now, she'd, should, shouldn, once, an, after, by, mustn't, between, of, couldn't, hadn, on, t, re, that, both, his, or, below, haven't, do, hasn, my, o, their, through, m, they, not, we're, theirs, her, she's, all, some, you, few, hasn't, which, doesn, this, themselves, until, when, those, doing, don, further, don't, have, mightn, off, other, isn, me, too, weren, am, been, can, i've, be, we've, aren't, a, you've, at, to, they're, does, wouldn't, i'm, only, being, because, before, up, isn't, had, we, you'll, each, is, no, so, are, shouldn't, myself, that'll, these, will, yourself, wasn, under, she, didn't, while, our, it's, needn't, wasn't, just, and, herself, they'll, yours, he, haven, its, himself, as, if, it, y, aren, she'll, same, were, he's, d, ours, over, about, for, he'll, what, did, but, ve, into, down, they'd, won, from, out, mustn, has, him, more, it'd, why, should've, the, you'd, shan, here, such, yourselve

In [15]:
# doing the same for the german ones
", ".join(stopwords_de)


'damit, ihre, nicht, sehr, mich, dasselbe, jenem, einmal, ihm, ander, an, eines, mancher, welchem, unserem, solchen, doch, da, derer, sie, aller, ihren, waren, bist, jenes, würden, eurem, für, einem, mein, einen, dessen, zur, einiges, deine, eurer, hat, diesem, die, diese, wollen, ihres, sonst, indem, meinen, unseren, vor, dein, manches, wo, welcher, ihr, jetzt, seinem, seinen, dieselben, zum, muss, meinem, sondern, deiner, deines, derselbe, wirst, mir, andern, am, auch, deinen, manchen, hatten, welche, anderr, bis, habe, solche, dass, und, zwar, daß, ihrer, solchem, jenen, als, wenn, anderes, unser, anders, könnte, jede, kein, soll, ist, keines, seiner, allem, unsere, meiner, mit, aber, jeder, während, dazu, andere, gegen, alle, nun, dieser, man, so, zu, anderer, das, deinem, nur, weiter, dies, warst, derselben, also, will, werde, ein, anderem, der, jener, werden, dich, dir, von, wird, wie, keinem, hin, anderm, dieses, euch, auf, ihn, zwischen, keinen, sich, bei, gewesen, ich, ohne, d

We now want to remove the stopwords of our stemmed documents for further processing

Again, we need a function to apply the stopword removal to each document

In [16]:
def remove_stopwords(stemmed_document: str, stopwords: Set) -> List[str]:
    # Define a helper function that returns True if the word is NOT a stopword
    def is_stopword(word):
        return not word in stopwords

    # Filter out stopwords from the stemmed document
    # Assumes `stemmed_document` is a list of stemmed words
    return list(filter(is_stopword, stemmed_document))


articles_en_final = [remove_stopwords(doc, stopwords_en) for doc in articles_en_stemmed]
articles_de_final = [remove_stopwords(doc, stopwords_de) for doc in articles_de_stemmed]

In [17]:
# print an article in all its versions. from the original over the tokenized, stemmed and final one
print(articles_en[0])
print(articles_en_tokenized[0])
print(articles_en_stemmed[0])
print(articles_en_final[0])

Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
['Reuters', '-', 'Short-sellers', ',', 'Wall', 'Street', "'s", 'dwindling\\band', 'of', 'ultra-cynics', ',', 'are', 'seeing', 'green', 'again', '.']
['reuter', '-', 'short-sel', ',', 'wall', 'street', "'s", 'dwindling\\band', 'of', 'ultra-cyn', ',', 'are', 'see', 'green', 'again', '.']
['reuter', '-', 'short-sel', ',', 'wall', 'street', "'s", 'dwindling\\band', 'ultra-cyn', ',', 'see', 'green', '.']


# Gensim

https://radimrehurek.com/gensim/

Gensim describes itself as "Topic Modelling for Humans".

We will use our NLTK-preprocessed documents as input to:

1. Build a dictionary – Mapping words to unique IDs.
2. Create a corpus – Representing documents as bag-of-words vectors.
3. Construct an index – Enabling efficient similarity queries.
4. Compute the TF-IDF matrix – Weighting terms by importance.
5. Run text queries – Searching for similar documents based on content.

This workflow demonstrates how Gensim transforms preprocessed text into powerful representations for topic modeling and similarity analysis.

In [ ]:
# Gensim is not installed by default in some environments, so we install it here
%pip install gensim


Note: you may need to restart the kernel to use updated packages.


## Building an TF-IDF model

Use a TF-IDF model to compare a user-provided input string against a trained corpus of English documents and retrieve the most similar articles.


In [18]:
from gensim import corpora, models, similarities

# Limit the number of documents to avoid memory issues with large models
size = 500  # Adjust this value if the model is too large to run efficiently

# Create a dictionary from the first `size` documents
# The dictionary maps each word to a unique ID
corpus_dictionary_en = corpora.Dictionary(articles_en_final[:size])

# Convert each document to a Bag-of-Words (BoW) representation
# Each document becomes a list of (word_id, frequency) tuples
corpus_en = [corpus_dictionary_en.doc2bow(document) for document in articles_en_final[:size]]

# Train a TF-IDF model on the BoW corpus
# This model assigns weights to words based on their importance across documents
model_en = models.TfidfModel(corpus_en)

# Create a similarity index using the TF-IDF weighted corpus
# This allows fast similarity queries between documents
index_en = similarities.MatrixSimilarity(model_en[corpus_en])

To calculate the similarity of an input, it has to be preprocessed the same way as our training data

In [ ]:
def query_en(query_string, model_en=model_en, index_en=index_en, stopwords_en=stopwords_en, size=size, corpus_dictionary_en=corpus_dictionary_en) -> List[Tuple[int, float]]:
    
    # Tokenize the query string using NLTK
    tokens = nltk.word_tokenize(query_string)

    # Apply stemming and stopword removal to the tokenized query
    # Assumes stopwords_en is a predefined set of English stopwords
    processed_query = remove_stopwords(
        stem(tokens, language="english"),
        stopwords_en
    )

    # Convert the processed query into a Bag-of-Words representation
    q = corpus_dictionary_en.doc2bow(processed_query)

    # Transform the BoW query into TF-IDF space
    q_model = model_en[q]

    # Compute similarity scores between the query and all documents
    result = index_en[q_model]

    # Sort results by similarity score in descending order
    result = sorted(enumerate(result), key=lambda item: -item[1])

    # Print the top 3 most similar documents and their scores
    for i, j in enumerate(result):
        if i > 2:
            break
        print(j, articles_en[:size][j[0]])

    # Return the full list of similarity scores with document indices
    return result

NameError: name 'corpus_dictionary' is not defined

Gensim returns the resulting document and its similarity

In [ ]:
query_en("Scientists United States", model_en, index_en);
# returns a list of tuples (document_idx, similarity_score) and prints (idx, score, article)